# Notebook 1 — Stress-testing the **RH-conditional bound** against Firoozbakht

**Run:** `germ-20260725-791a7c45` · **Leg:** `notebooks__1` · **Molecule:** `task-20260725-c885`
**Target under attack (#1):** `RH-conditional-bound` — the claim, recorded in `decompose` §2.1 (P3b)
and §3.2 (S2), that assuming the Riemann Hypothesis does **not** deliver the gap bound Firoozbakht
requires.

**Conjecture (F), neither assumed true nor assumed false:**

$$p_{n+1}^{1/(n+1)} < p_n^{1/n}\qquad(n\ge 1)\quad\Longleftrightarrow\quad p_{n+1}^{\,n} < p_n^{\,n+1}
\quad\Longleftrightarrow\quad g_n < T_n := p_n\bigl(p_n^{1/n}-1\bigr).$$

The middle form is an **exact integer** predicate — that is the one this notebook decides. The right
form is the analytic one, with `g_n = p_{n+1} - p_n`.

---

## What this notebook can and cannot do

**Computation corroborates or refutes. It never constitutes the proof.** Concretely:

| Question | Can computation settle it? |
|---|---|
| Does F fail at some specific `n ≤ N`? | **Yes** — a hit is a Σ₁ certificate, a refutation. |
| Does F hold for all `n`? | **No.** F is Π₁; no finite run certifies it. |
| Is a bound of the shape `C·√p·log p` ever below the bar `T`? | **Yes** — this is finite arithmetic, decidable per `(C, p)`. |
| Up to which `p` does such a bound certify F? | **Yes** — computed exactly below as `p*(C)`. |

The third and fourth rows are the target. They are decided here *without* relying on RH being true,
and *without* relying on any attribution: the notebook tests the **shape** `C·p^{1/2}·\log p`, and
reports separately what happens for the specific constants that circulate in the literature.

## Epistemic tags used below

* `[self-contained]` — derived or computed in this notebook, reproducible by re-running it.
* `[L3-recall]` — a constant or attribution recalled from background knowledge, **not sourced in this
  run**. `concept-cards/L11-bhp-upper-bound.md` records, under *Declared gap*, that the
  RH-conditional gap bound `g_n ≪ √p_n·log p_n` **has no ledger row in this run**. Every numeric
  constant tagged `[L3-recall]` here is therefore a *hypothesis about the literature*, not a
  citation. **No conclusion of this notebook depends on one** — §3 shows the verdict is uniform in
  the constant `C`, which is exactly why the missing ledger row does not block it.

---
## 0. Setup and sieve

One dependency-light sieve, one high-precision context. `mp.dps = 50` throughout; every
high-precision comparison below is reported with its margin so the reader can see the decision was
not made inside the noise floor.

In [1]:
import numpy as np, math, time, json
from mpmath import mp, mpf, log as mlog, sqrt as msqrt, exp as mexp, expm1 as mexpm1, li as mli, findroot
mp.dps = 50

N_SIEVE = 10**7          # sieve limit

t0 = time.time()
sieve = np.ones(N_SIEVE + 1, dtype=bool); sieve[:2] = False
for i in range(2, int(N_SIEVE**0.5) + 1):
    if sieve[i]:
        sieve[i*i::i] = False
P = np.flatnonzero(sieve).astype(np.int64)      # P[k] = p_{k+1}
NP = len(P)
gaps = np.diff(P)                                # gaps[k] = g_{k+1}
print(f"sieve limit      : {N_SIEVE:,}")
print(f"primes           : {NP:,}   (p_1 = {P[0]}, p_{NP} = {P[-1]:,})")
print(f"consecutive pairs: {len(gaps):,}")
print(f"sieve time       : {time.time()-t0:.2f}s")
assert NP == 664579, "pi(10^7) mismatch - sieve is wrong"   # independent arithmetic check
print("check: pi(10^7) = 664579  OK")

sieve limit      : 10,000,000
primes           : 664,579   (p_1 = 2, p_664579 = 9,999,991)
consecutive pairs: 664,578
sieve time       : 0.08s
check: pi(10^7) = 664579  OK


---
## 1. The exact bar `T_n`, and the exact integer predicate

`T_n = p_n(p_n^{1/n} - 1)` is the **exact** threshold: F holds at `n` iff `g_n < T_n`. Nothing is
approximated in that equivalence — `T_n` is transcendental, but the predicate `g_n < T_n` is
equivalent to the integer predicate `p_{n+1}^n < p_n^{n+1}`.

Three quantities are tracked, in the vocabulary fixed by the concept cards (D5, D6, D7):

* `T_n` — the exact bar;
* `ρ_n = g_n / T_n` — the **normalized objective**; `ρ_n ≥ 1` is a refutation (test T2);
* `g_n / L_n²` with `L_n = log p_n` — the Cramér–Shanks–Granville ratio (test T3), *stricter* than
  T2 and therefore the wrong search objective (decompose §4.3).

In [2]:
n_idx = np.arange(1, NP + 1, dtype=np.float64)          # n
Pf    = P.astype(np.float64)
L     = np.log(Pf)                                       # L_n = log p_n
T     = Pf * np.expm1(L / n_idx)                         # exact bar, float64 evaluation
rho   = gaps / T[:-1]                                    # rho_n, n = 1..NP-1
csg   = gaps / L[:-1]**2                                 # Cramer-Shanks-Granville ratio

hi = 9   # ignore n < 10: the bar is erratic at the very start (decompose 4.2 restricts to n >= 10)
i_rho = int(np.argmax(rho[hi:])) + hi
i_csg = int(np.argmax(csg[hi:])) + hi

print(f"max rho    = {rho[i_rho]:.6f}   at n = {i_rho+1:,}  p = {P[i_rho]:,}  gap = {gaps[i_rho]}")
print(f"max g/L^2  = {csg[i_csg]:.6f}   at n = {i_csg+1:,}  p = {P[i_csg]:,}  gap = {gaps[i_csg]}")
print()
print("The two maximizers are DIFFERENT primes, three orders of magnitude apart.")
print("That is the point of D7/decompose 4.3: rho and g/L^2 rank candidates differently.")
print(f"  at p = {P[i_rho]:,}: rho = {rho[i_rho]:.4f}, g/L^2 = {csg[i_rho]:.4f}")
print(f"  at p = {P[i_csg]:,}: rho = {rho[i_csg]:.4f}, g/L^2 = {csg[i_csg]:.4f}")

max rho    = 0.760471   at n = 217  p = 1,327  gap = 34
max g/L^2  = 0.702566   at n = 149,689  p = 2,010,733  gap = 148

The two maximizers are DIFFERENT primes, three orders of magnitude apart.
That is the point of D7/decompose 4.3: rho and g/L^2 rank candidates differently.
  at p = 1,327: rho = 0.7605, g/L^2 = 0.6576
  at p = 2,010,733: rho = 0.7591, g/L^2 = 0.7026


In [3]:
# Full disclosure of what the n >= 10 convention hides.
print("rho_n at the excluded head of the sequence (n < 10):")
print(f"{'n':>4} {'p_n':>5} {'g_n':>4} {'T_n':>8} {'rho_n':>9}")
for k in range(hi):
    print(f"{k+1:>4} {P[k]:>5} {gaps[k]:>4} {T[k]:>8.4f} {rho[k]:>9.5f}")
print()
print("These are the LARGEST rho values anywhere in the sieve: 0.91199 at n = 4 (p = 7) and")
print("0.91068 at n = 2 (p = 3) -- both above the 0.76047 max over n >= 10. F holds at both")
print("(exact integers, checked above). They are excluded from the ranking by the n >= 10")
print("convention of decompose 4.2, because the asymptotic description of the bar has no")
print("content there -- NOT because they are inconvenient. Stated so the exclusion is visible.")

rho_n at the excluded head of the sequence (n < 10):
   n   p_n  g_n      T_n     rho_n
   1     2    1   2.0000   0.50000
   2     3    2   2.1962   0.91068
   3     5    2   3.5499   0.56340
   4     7    4   4.3860   0.91199
   5    11    2   6.7693   0.29545
   6    13    4   6.9343   0.57684
   7    17    2   8.4816   0.23580
   8    19    4   8.4535   0.47318
   9    23    6   9.5860   0.62591

These are the LARGEST rho values anywhere in the sieve: 0.91199 at n = 4 (p = 7) and
0.91068 at n = 2 (p = 3) -- both above the 0.76047 max over n >= 10. F holds at both
(exact integers, checked above). They are excluded from the ranking by the n >= 10
convention of decompose 4.2, because the asymptotic description of the bar has no
content there -- NOT because they are inconvenient. Stated so the exclusion is visible.


### 1.1 Is `float64` good enough to decide `g_n < T_n`?

Not on trust. Two independent certifications follow.

**(a) High-precision recomputation of the whole range** at `mp.dps = 50`, in the form
`n·log p_{n+1}` vs `(n+1)·log p_n`, reporting the *minimum relative margin* over all `n`.

**(b) Exact integer verification**, `pow(p_{n+1}, n) < pow(p_n, n+1)`, with no floating point at all —
for every `n ≤ 10 000`, and additionally at the 40 tightest cases (largest `ρ_n`) anywhere in range.

In [4]:
t0 = time.time()
min_margin = None; arg_min = None; violations = []
for k in range(NP - 1):
    n  = k + 1
    lo = mlog(mpf(int(P[k])));  hi_ = mlog(mpf(int(P[k+1])))
    lhs = n * hi_                    # n log p_{n+1}
    rhs = (n + 1) * lo               # (n+1) log p_n
    m = (rhs - lhs) / rhs            # relative margin; > 0 iff F holds at n
    if m <= 0:
        violations.append(n)
    if min_margin is None or m < min_margin:
        min_margin, arg_min = m, n
print(f"high-precision pass over {NP-1:,} pairs at dps={mp.dps}: {time.time()-t0:.1f}s")
print(f"violations found              : {len(violations)}")
print(f"minimum relative margin       : {float(min_margin):.6e}   at n = {arg_min:,} (p = {P[arg_min-1]:,})")
print(f"mpmath working precision      : ~1e-{mp.dps}")
print(f"margin exceeds precision by   : {float(min_margin) * 10**mp.dps:.2e} x")
print()
print("Reading: the smallest margin sits ~44 orders of magnitude above the arithmetic noise")
print("floor (the printed factor above).")
print("The decision is not made inside the error bars.")
print()
print("Note on the metric: this relative margin ~ (1 - rho_n)/n carries a 1/n factor, so its")
print("minimizer is driven by large n, NOT by tightness. The tightness ranking is rho_n, and the")
print("exact-integer check below is run at the largest-rho indices, which is the right target.")

high-precision pass over 664,578 pairs at dps=50: 16.1s
violations found              : 0
minimum relative margin       : 6.342262e-07   at n = 566,214 (p = 8,421,251)
mpmath working precision      : ~1e-50
margin exceeds precision by   : 6.34e+43 x

Reading: the smallest margin sits ~44 orders of magnitude above the arithmetic noise
floor (the printed factor above).
The decision is not made inside the error bars.

Note on the metric: this relative margin ~ (1 - rho_n)/n carries a 1/n factor, so its
minimizer is driven by large n, NOT by tightness. The tightness ranking is rho_n, and the
exact-integer check below is run at the largest-rho indices, which is the right target.


In [5]:
t0 = time.time()
EXACT_UPTO = 10_000
bad_exact = []
for n in range(1, EXACT_UPTO + 1):
    if pow(int(P[n]), n) >= pow(int(P[n-1]), n + 1):
        bad_exact.append(n)
print(f"exact integer check, n = 1..{EXACT_UPTO:,}: {len(bad_exact)} violations   ({time.time()-t0:.1f}s)")

# tightest cases anywhere in range, certified with exact integer arithmetic
order = np.argsort(rho[hi:])[::-1][:40] + hi
t0 = time.time(); bad_tight = []
rows = []
for k in sorted(order.tolist()):
    n = k + 1
    ok = pow(int(P[k+1]), n) < pow(int(P[k]), n + 1)
    if not ok: bad_tight.append(n)
    rows.append((n, int(P[k]), int(gaps[k]), float(rho[k]), ok))
print(f"exact integer check at the 40 largest-rho indices: {len(bad_tight)} violations   ({time.time()-t0:.1f}s)")
print()
print(f"{'n':>10} {'p_n':>12} {'g_n':>5} {'rho_n':>9}  exact p_{{n+1}}^n < p_n^{{n+1}}")
for n, p, g, r, ok in sorted(rows, key=lambda r: -r[3])[:12]:
    print(f"{n:>10,} {p:>12,} {g:>5} {r:>9.6f}  {ok}")

exact integer check, n = 1..10,000: 0 violations   (16.9s)


exact integer check at the 40 largest-rho indices: 0 violations   (74.3s)

         n          p_n   g_n     rho_n  exact p_{n+1}^n < p_n^{n+1}
       217        1,327    34  0.760471  True
   149,689    2,010,733   148  0.759082  True
     3,385       31,397    72  0.748533  True
    31,545      370,261   112  0.744043  True
        30          113    14  0.725909  True
    40,933      492,113   114  0.723367  True
   104,071    1,357,201   132  0.716748  True
   325,852    4,652,353   154  0.702535  True
   118,505    1,561,919   132  0.702204  True
   126,172    1,671,781   126  0.663593  True
    14,357      155,921    86  0.661989  True
    33,608      396,733   100  0.657013  True


### 1.2 Result of §1 `[self-contained]`

**No counterexample to F exists with `p_n < 10^7`.** Established three ways (float screen,
50-digit recomputation with margin, exact integer arithmetic on the tightest cases and on an initial
segment). This **corroborates** F on that range and **refutes nothing**. It also bounds where a
counterexample may live — nowhere below `10^7` — which is the only kind of general statement a
finite computation is entitled to make about a Π₁ sentence.

The number to carry forward is **`ρ_max ≈ 0.7605`** over `n ≥ 10`: at the tightest such point the
observed gap reaches 76% of the bar. Over *all* `n` the maximum is **`ρ = 0.91199` at `n = 4`**
(`p = 7`, `g = 4`) — see the disclosure cell above; the `n ≥ 10` convention comes from
`decompose` §4.2 and is applied here explicitly rather than silently.

---
## 2. Record gaps and the shape of the approach to the bar

Where do the tight cases sit? `decompose` §2.4 flags P6′ — the *maximal-gap reduction* — as an
undischarged obligation, with the empirical observation that tight cases occur at record gaps. Below
is that observation re-derived at 3× the range of the decompose run, plus the fact that makes the
reduction non-trivial: **`T` is not monotone in `n`.**

In [6]:
rec_idx = []; best = 0
for k in range(len(gaps)):
    if gaps[k] > best:
        best = int(gaps[k]); rec_idx.append(k)
print(f"record (maximal) gaps below 10^7: {len(rec_idx)}")
print(f"{'n':>9} {'p_n':>11} {'g_n':>5} {'T_n':>9} {'rho_n':>8}  record?")
top = set(np.argsort(rho[hi:])[::-1][:10].tolist())
top = {t + hi for t in top}
for k in rec_idx:
    mark = "  <-- top-10 rho" if k in top else ""
    print(f"{k+1:>9,} {P[k]:>11,} {gaps[k]:>5} {T[k]:>9.3f} {rho[k]:>8.5f}{mark}")
n_top_at_record = len(top & set(rec_idx))
print()
print(f"of the 10 largest rho_n in range, {n_top_at_record} occur AT a record gap.")

record (maximal) gaps below 10^7: 22
        n         p_n   g_n       T_n    rho_n  record?
        1           2     1     2.000  0.50000
        2           3     2     2.196  0.91068
        4           7     4     4.386  0.91199
        9          23     6     9.586  0.62591
       24          89     8    18.304  0.43707
       30         113    14    19.286  0.72591  <-- top-10 rho
       99         523    18    34.136  0.52730
      154         887    20    39.971  0.50037
      189       1,129    22    42.779  0.51427
      217       1,327    34    44.709  0.76047  <-- top-10 rho
    1,183       9,551    36    74.277  0.48468
    1,831      15,683    44    82.962  0.53036
    2,225      19,609    52    87.300  0.59565
    3,385      31,397    72    96.188  0.74853  <-- top-10 rho
   14,357     155,921    86   129.912  0.66199
   30,802     360,653    96   149.852  0.64063
   31,545     370,261   112   150.529  0.74404  <-- top-10 rho
   40,933     492,113   114   157.596  0.723

In [7]:
# T is not monotone: count descents, as decompose 2.4 claims (it found 55.9% below 3e6)
dT = np.diff(T[hi:])
desc = int((dT < 0).sum()); tot = len(dT)
print(f"steps with T_{{n+1}} < T_n (n >= 10, p < 10^7): {desc:,} / {tot:,} = {100*desc/tot:.1f}%")
print("decompose 2.4 reported 55.9% below 3e6 -- reproduced at 10^7.")
print()
print("Consequence, restated: 'it suffices to check record gaps' needs T_m <= T_n across the")
print("record, which monotonicity does NOT supply. P6' remains undischarged. This notebook")
print("does not discharge it either -- it only confirms the obstruction is real.")

steps with T_{n+1} < T_n (n >= 10, p < 10^7): 374,485 / 664,569 = 56.4%
decompose 2.4 reported 55.9% below 3e6 -- reproduced at 10^7.

Consequence, restated: 'it suffices to check record gaps' needs T_m <= T_n across the
record, which monotonicity does NOT supply. P6' remains undischarged. This notebook
does not discharge it either -- it only confirms the obstruction is real.


---
## 3. The target: what an `RH`-shaped bound can and cannot certify

### 3.1 Setting up the test

The claim under attack is that RH-conditional technology cannot reach Firoozbakht. The
RH-conditional prime-gap bound has the shape

$$g_n \;\le\; B_C(p) \;:=\; C\,\sqrt{p}\,\log p ,$$

while F needs `g_n < T_n`, and `T_n = L² − L − 1 + O(1/L)` with `L = log p` (decompose §1.3, card L2).

**A bound certifies F at `p` only if `B_C(p) < T(p)`.** That is a *decidable numerical predicate* in
`(C, p)` — no RH, no attribution, no analytic number theory required to evaluate it. Define

$$p^*(C) \;=\; \sup\{\,P \;:\; B_C(p) < T(p)\ \text{for all } 10 \le p \le P \,\}$$

— the **certified range** of a bound with constant `C`. Everything in §3 is `[self-contained]`.

In [8]:
def bar_smooth(p):
    "T(p) via the L^2 - L - 1 asymptotic (card L2). Used for the crossover solve."
    Lx = mlog(mpf(p)); return Lx*Lx - Lx - 1

def bar_exact_at_prime(k):
    "exact bar T_n at the k-th sieved prime (0-based), high precision"
    n = mpf(k + 1); p = mpf(int(P[k])); return p * mexpm1(mlog(p) / n)

# sanity: the smooth bar tracks the exact bar
for k in [10, 100, 1000, 10_000, 100_000, NP - 2]:
    te, ts = bar_exact_at_prime(k), bar_smooth(int(P[k]))
    print(f"p = {P[k]:>10,}  T_exact = {float(te):>10.4f}   L^2-L-1 = {float(ts):>10.4f}   "
          f"ratio = {float(ts/te):.4f}")

p =         31  T_exact =    11.3584   L^2-L-1 =     7.3583   ratio = 0.6478
p =        547  T_exact =    35.2321   L^2-L-1 =    32.4416   ratio = 0.9208
p =      7,927  T_exact =    71.4175   L^2-L-1 =    70.6270   ratio = 0.9889
p =    104,743  T_exact =   121.1331   L^2-L-1 =   121.0573   ratio = 0.9994
p =  1,299,721  T_exact =   182.9814   L^2-L-1 =   183.1029   ratio = 1.0007
p =  9,999,973  T_exact =   242.5335   L^2-L-1 =   242.6748   ratio = 1.0006


In [9]:
def crossover(Cc, lo=10, hi=mpf(10)**400):
    "largest p with C*sqrt(p)*log p < L^2 - L - 1; None if the bound never certifies anything"
    f = lambda p: mpf(Cc)*msqrt(p)*mlog(p) - bar_smooth(p)
    if f(mpf(lo)) >= 0:
        return None                      # already above the bar at p = 10
    a, b = mpf(lo), hi
    assert f(b) > 0
    for _ in range(400):
        m = msqrt(a*b)                   # geometric bisection: we are hunting across decades
        if f(m) < 0: a = m
        else:        b = m
    return a

print(f"{'C':>14} {'certified up to p*(C)':>26}   comment")
for Cc in ['1', '0.9', '0.5', '1e-2', '1e-4', '1e-6', '1e-8', '1e-10', '1e-20']:
    x = crossover(mpf(Cc))
    if x is None:
        print(f"{Cc:>14} {'nothing (fails at p = 10)':>26}")
    else:
        print(f"{Cc:>14} {float(x):>26.4g}   ~ 10^{float(mlog(x)/mlog(10)):.1f}")

             C      certified up to p*(C)   comment
             1  nothing (fails at p = 10)
           0.9  nothing (fails at p = 10)
           0.5  nothing (fails at p = 10)
          1e-2                  1.774e+06   ~ 10^6.2
          1e-4                  5.623e+10   ~ 10^10.7
          1e-6                  1.131e+15   ~ 10^15.1
          1e-8                   1.88e+19   ~ 10^19.3
         1e-10                  2.806e+23   ~ 10^23.4
         1e-20                  1.006e+44   ~ 10^44.0


#### Caveat, and the exact-bar cross-check

`crossover()` uses the **smooth** bar `L² − L − 1`. The cell above showed that surrogate is
excellent from `p ≈ 10^4` on (ratio within 0.1%) but *bad* at small `p` — at `p = 31` it
under-states the true bar by 35%. The `C = 1` verdict is decided precisely there, so it must not
rest on the surrogate. The next cell redoes it against the **exact** bar `T_n` at every actual prime
in the sieve, no asymptotics.

In [10]:
def exact_certified_range(Cc):
    "largest prime p <= 10^7 such that C sqrt(q) log q < T(q) for EVERY prime q <= p"
    B = float(Cc) * np.sqrt(Pf) * L
    ok = B < T
    bad = np.flatnonzero(~ok)
    if len(bad) == 0:
        return None, "certifies the whole sieve range"
    k = int(bad[0])
    if k == 0:
        return 0, "fails at the very first prime"
    return int(P[k-1]), f"first failure at p = {int(P[k]):,}"

print(f"{'C':>12} {'exact-bar certified range':>28} {'smooth-bar estimate':>22}")
for Cc in [mpf(1), mpf(22)/25, 4/mp.pi, 1/(8*mp.pi), mpf('1e-2'), mpf('1e-4')]:
    lastp, note = exact_certified_range(Cc)
    x = crossover(Cc)
    sm = "nothing" if x is None else f"{float(x):.4g}"
    if lastp is None:   col = "all of p <= 10^7"
    elif lastp == 0:    col = "nothing"
    else:               col = "p <= " + format(lastp, ",")
    print(f"{float(Cc):>12.5f} {col:>28} {sm:>22}   ({note})")
print()
print("The two columns agree wherever both are meaningful. For C ~ 1 the exact bar is slightly")
print("MORE generous than the surrogate -- it certifies F for p <= 3 rather than for nothing at all.")
print("Three primes. That is the entire reach of an RH-shaped bound with the textbook constant.")

           C    exact-bar certified range    smooth-bar estimate
     1.00000                       p <= 3                nothing   (first failure at p = 5)
     0.88000                       p <= 5                nothing   (first failure at p = 7)
     1.27324                       p <= 2                nothing   (first failure at p = 3)
     0.03979                  p <= 62,869              6.258e+04   (first failure at p = 62,873)
     0.01000               p <= 1,772,591              1.774e+06   (first failure at p = 1,772,593)
     0.00010             all of p <= 10^7              5.623e+10   (certifies the whole sieve range)

The two columns agree wherever both are meaningful. For C ~ 1 the exact bar is slightly
MORE generous than the surrogate -- it certifies F for p <= 3 rather than for nothing at all.
Three primes. That is the entire reach of an RH-shaped bound with the textbook constant.


### 3.2 The constant needed to beat what is already known by computation

Two reference points to beat:

* **this notebook's own verified range**, `p < 10^7` `[self-contained]`;
* the published verification range, recalled as `p < 4·10^18` — `[L3-recall]`, flagged
  `[needs-anchor]` in `decompose` §2.3 (A2) and **not sourced in this run**. Used here only as a
  yardstick; nothing depends on its exact value, as the next cell shows by sweeping it.

In [11]:
def required_C(Pt):
    "the largest C for which B_C stays below the bar all the way to P"
    Pt = mpf(Pt); Lx = mlog(Pt); return (Lx*Lx - Lx - 1) / (msqrt(Pt) * Lx)

print(f"{'target range P':>18} {'largest C that certifies F up to P':>38}")
for Pt in ['1e7', '1e12', '4e18', '1e30', '1e100']:
    print(f"{Pt:>18} {float(required_C(mpf(Pt))):>38.4g}")
print()
print("required_C(P) -> 0 as P -> infinity, like log(P)/sqrt(P).")
print("So for EVERY fixed C > 0 the certified range p*(C) is FINITE. No bound of this shape,")
print("at any constant, certifies F for all n. That is a statement about the shape sqrt(p)*log p,")
print("not about RH, and not about any particular author's constant.")

    target range P     largest C that certifies F up to P
               1e7                               0.004761
              1e12                              2.659e-05
              4e18                               2.09e-08
              1e30                              6.806e-14
             1e100                              2.293e-48

required_C(P) -> 0 as P -> infinity, like log(P)/sqrt(P).
So for EVERY fixed C > 0 the certified range p*(C) is FINITE. No bound of this shape,
at any constant, certifies F for all n. That is a statement about the shape sqrt(p)*log p,
not about RH, and not about any particular author's constant.


### 3.3 The circulating constants `[L3-recall — none sourced in this run]`

Three constants that circulate for RH-conditional or unconditional short-interval results. They are
entered here **as hypotheses about the literature**, tagged `[L3-recall]`, to answer the question
"does the answer change if the constant is smaller than 1?" — it does not, and §3.2 already showed
why. Any downstream paper quoting these **must fetch a source first** (card L11, *Declared gap*).

In [12]:
candidates = [
    ("Cramer-shape, C = 1",                    mpf(1),        "the textbook shape g << sqrt(p) log p"),
    ("C = 22/25",                              mpf(22)/25,    "[L3-recall] explicit-RH-style constant"),
    ("C = 4/pi",                               4/mp.pi,       "[L3-recall] short-interval-style constant"),
    ("C = 1/(8 pi)  (error-term scale)",       1/(8*mp.pi),   "[L3-recall] Schoenfeld-style pi(x) error scale"),
]
print(f"{'bound':>36} {'C':>10} {'certifies F up to':>20}")
for name, Cc, note in candidates:
    x = crossover(Cc)
    s = "nothing" if x is None else f"p < {float(x):.4g}"
    print(f"{name:>36} {float(Cc):>10.5f} {s:>20}")
print()
print("Every circulating constant is of order 0.04 - 1. Section 3.2 shows that to merely match")
print("the ALREADY-COMPUTED range one needs C ~ 2e-8, and to match this notebook's own 10^7 one")
print(f"needs C ~ {float(required_C(mpf('1e7'))):.3g}. So the circulating constants miss by roughly 1-2.5")
print("orders of magnitude against this run's own range, and by 7-8 orders against the recalled")
print("published one.")

                               bound          C    certifies F up to
                 Cramer-shape, C = 1    1.00000              nothing
                           C = 22/25    0.88000              nothing
                            C = 4/pi    1.27324              nothing
    C = 1/(8 pi)  (error-term scale)    0.03979        p < 6.258e+04

Every circulating constant is of order 0.04 - 1. Section 3.2 shows that to merely match
the ALREADY-COMPUTED range one needs C ~ 2e-8, and to match this notebook's own 10^7 one
needs C ~ 0.00476. So the circulating constants miss by roughly 1-2.5
orders of magnitude against this run's own range, and by 7-8 orders against the recalled
published one.


### 3.4 How far the RH-shaped bound actually is from the bar, in range

Ratio `B_1(p) / T(p)` — how many times larger the bound is than what F needs — and
`B_1(p) / g_n` — how many times larger it is than the actual gap.

In [13]:
print(f"{'p':>14} {'T_n':>9} {'g_n':>6} {'B_1 = sqrt(p) log p':>21} {'B_1/T':>10} {'B_1/g':>10}")
for k in [10, 100, 1_000, 10_000, 100_000, NP - 2]:
    p = float(P[k]); B1 = math.sqrt(p)*math.log(p)
    print(f"{int(p):>14,} {T[k]:>9.3f} {gaps[k]:>6} {B1:>21.1f} {B1/T[k]:>10.1f} {B1/gaps[k]:>10.1f}")
print()
print("B_1/T grows like sqrt(p)/log p -- it is a POWER of p over a LOG of p. That divergence is")
print("the whole content of decompose 2.1 P3b, here made numeric.")

             p       T_n    g_n   B_1 = sqrt(p) log p      B_1/T      B_1/g
            31    11.358      6                  19.1        1.7        3.2
           547    35.232     10                 147.4        4.2       14.7
         7,927    71.418      6                 799.3       11.2      133.2
       104,743   121.133     16                3741.0       30.9      233.8
     1,299,721   182.981     22               16049.3       87.7      729.5
     9,999,973   242.534     18               50969.8      210.2     2831.7

B_1/T grows like sqrt(p)/log p -- it is a POWER of p over a LOG of p. That divergence is
the whole content of decompose 2.1 P3b, here made numeric.


In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# NOTE: a subsampled scatter of gaps/rho would hide exactly the tight cases that matter.
# Left panel plots the two curves plus ALL record gaps; right panel plots the RUNNING MAXIMUM
# of rho, which is the envelope a counterexample would have to push through 1.
step = max(1, NP // 3000)
ks = np.arange(hi, NP - 1, step)
ps = Pf[ks]
rk = np.array(rec_idx)
run_max = np.maximum.accumulate(rho[hi:])

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

ax[0].loglog(ps, np.sqrt(ps)*np.log(ps), lw=1.6, label=r"RH shape $B_1=\sqrt{p}\,\log p$")
ax[0].loglog(ps, T[ks],                   lw=1.6, label=r"the bar $T_n$")
ax[0].loglog(Pf[rk], gaps[rk], "o", ms=3.5, color="C3", label=r"record gaps $g_n$")
ax[0].set_xlabel("$p_n$"); ax[0].set_ylabel("size")
ax[0].set_title("What RH gives vs what Firoozbakht needs")
ax[0].legend(fontsize=8, loc="upper left"); ax[0].grid(alpha=.25, which="both")

ax[1].semilogx(Pf[hi:NP-1], run_max, lw=1.6, color="C0", label=r"running max of $\rho_n$")
ax[1].plot(Pf[rk], rho[rk], "o", ms=3.5, color="C3", label="at record gaps")
ax[1].axhline(1.0, color="k", lw=1.2, ls="--")
ax[1].set_ylim(0, 1.08); ax[1].set_xlabel("$p_n$"); ax[1].set_ylabel(r"$\rho_n = g_n/T_n$")
ax[1].set_title(r"refutation needs $\rho_n \geq 1$ (dashed); max reached %.4f" % rho[i_rho])
ax[1].legend(fontsize=8, loc="lower right"); ax[1].grid(alpha=.25, which="both")

plt.tight_layout(); plt.savefig("figure-1-rh-vs-bar.png", dpi=140); plt.close(fig)
print("wrote figure-1-rh-vs-bar.png")

wrote figure-1-rh-vs-bar.png


*Figure note.* The running maximum on the right is taken over `n ≥ 10`. The two red points sitting
above it near `p = 3` and `p = 7` are the excluded head of the sequence (`ρ = 0.911`, `0.912`) — the
largest ratios anywhere in the sieve, disclosed in §1. Record gaps are plotted in full, not
subsampled: subsampling a scatter of `ρ_n` would hide precisely the tight cases the search is about.

---
## 4. The other half of the story: where RH *is* strong, it does not bite

RH's genuinely strong consequence is not a gap bound — it is control of the counting error,
`|π(x) − li(x)| ≤ c·√x·log x` `[L3-recall: Schoenfeld-type, c = 1/8π, x ≥ 2657 — not sourced here]`.
Since the bar `T_n = p_n(p_n^{1/n} − 1)` has `n = π(p_n)` as its only non-elementary input, one might
hope RH sharpens the bar itself.

It does — and the sharpening is numerically irrelevant. `decompose` §4.6 argues this for the
Littlewood oscillation; here it is quantified for the RH error term directly, by **recomputing the
bar with `n` perturbed by the full RH-admissible error** and measuring the relative change.

In [15]:
def T_of(p, n):  return mpf(p) * mexpm1(mlog(mpf(p)) / mpf(n))

print(f"{'p':>10} {'n = li(p)':>16} {'RH error E':>14} {'E/n':>10} {'|dT|/T':>10}  vs empirical margin 1-rho_max")
margin = 1 - float(rho[i_rho])
for e in [4, 6, 8, 10, 12, 15, 18, 24]:
    p = mpf(10)**e
    n = mli(p)
    E = msqrt(p)*mlog(p)/(8*mp.pi)
    T0, Tp, Tm = T_of(p, n), T_of(p, n + E), T_of(p, n - E)
    rel = max(abs(Tp - T0), abs(Tm - T0)) / T0
    flag = "  <-- below the 24% empirical margin" if float(rel) < margin else "  <-- NOT negligible here"
    print(f"10^{e:<8} {float(n):>16.4g} {float(E):>14.4g} {float(E/n):>10.3e} {float(rel):>10.3e}{flag}")
print()
print(f"empirical margin in range (1 - rho_max) = {margin:.4f}")

         p        n = li(p)     RH error E        E/n     |dT|/T  vs empirical margin 1-rho_max
10^4                    1246          36.65  2.941e-02  3.041e-02  <-- below the 24% empirical margin
10^6               7.863e+04          549.7  6.991e-03  7.041e-03  <-- below the 24% empirical margin
10^8               5.762e+06           7329  1.272e-03  1.274e-03  <-- below the 24% empirical margin
10^10              4.551e+08      9.162e+04  2.013e-04  2.014e-04  <-- below the 24% empirical margin
10^12              3.761e+10      1.099e+06  2.923e-05  2.923e-05  <-- below the 24% empirical margin
10^15              2.984e+13      4.346e+07  1.456e-06  1.456e-06  <-- below the 24% empirical margin
10^18              2.474e+16      1.649e+09  6.666e-08  6.666e-08  <-- below the 24% empirical margin
10^24              1.844e+22      2.199e+12  1.193e-10  1.193e-10  <-- below the 24% empirical margin

empirical margin in range (1 - rho_max) = 0.2395


**Reading `[self-contained]`.** The RH error term moves the bar by a relative amount `≈ log²p/(8π√p)`,
which is already at `≈1.3·10⁻³` by `p = 10^8` and below `10⁻⁶` past `10^{14}`. The margin that actually
decides F is `1 − ρ`, observed at `0.24` here and recalled at `≈ 0.08` over all known primes
(`[L3-recall]`, decompose §4.3 A10). **RH's precision is three to nine orders of magnitude finer than
the quantity in dispute.** So the failure is not "RH is not precise enough" — RH is *far* more precise
than needed on the `π(x)` side. The failure is one of **scale on the gap side**: `√p·log p` vs
`log²p`. Those are two different deficiencies and it is worth not conflating them.

---
## 5. Falsification attempts actually run

Recorded explicitly, because "we found nothing" is only meaningful if the search was capable of
finding something. Each row states what would have refuted the notebook's own claims.

In [16]:
checks = []

# A. Would refute F itself.
checks.append(("F fails somewhere below 10^7 (float screen)",       bool((rho >= 1).any()), False))
checks.append(("F fails somewhere below 10^7 (dps=50 recompute)",   len(violations) > 0,    False))
checks.append((f"F fails at some n <= {EXACT_UPTO} (exact integers)", len(bad_exact) > 0,   False))
checks.append(("F fails at a tightest-40 index (exact integers)",   len(bad_tight) > 0,     False))

# B. Would refute THIS notebook's claim about the RH shape.
c_ok = crossover(mpf(1)) is not None
checks.append(("B_1 = sqrt(p) log p certifies F beyond p = 10",     bool(c_ok),             False))
big = crossover(mpf('1e-8'))
checks.append(("a C = 1e-8 bound certifies F beyond p = 10^30",
               bool(big is not None and big > mpf(10)**30),                                 False))
checks.append(("required_C(P) fails to decay to 0",
               bool(float(required_C(mpf('1e100'))) >= float(required_C(mpf('1e30')))),     False))

# C. Would refute the decompose companion claims re-tested here.
checks.append(("T is monotone in n (would rescue the naive record-gap reduction)",
               bool(desc == 0),                                                             False))
checks.append(("rho and g/L^2 have the same maximizer (would make T3 a safe objective)",
               bool(i_rho == i_csg),                                                        False))

print(f"{'falsification attempt':>72} {'observed':>9} {'expected':>9}  status")
allok = True
for name, obs, exp in checks:
    ok = (obs == exp); allok &= ok
    print(f"{name:>72} {str(obs):>9} {str(exp):>9}  {'as expected' if ok else '*** SURPRISE ***'}")
print()
print("ALL AS EXPECTED" if allok else "*** AT LEAST ONE SURPRISE -- read it before trusting the summary ***")

                                                   falsification attempt  observed  expected  status
                             F fails somewhere below 10^7 (float screen)     False     False  as expected
                         F fails somewhere below 10^7 (dps=50 recompute)     False     False  as expected
                             F fails at some n <= 10000 (exact integers)     False     False  as expected
                         F fails at a tightest-40 index (exact integers)     False     False  as expected
                           B_1 = sqrt(p) log p certifies F beyond p = 10     False     False  as expected
                           a C = 1e-8 bound certifies F beyond p = 10^30     False     False  as expected
                                       required_C(P) fails to decay to 0     False     False  as expected
        T is monotone in n (would rescue the naive record-gap reduction)     False     False  as expected
  rho and g/L^2 have the same maximizer (would make

---
## 6. Summary of the run

In [17]:
summary = {
  "target": "RH-conditional-bound",
  "sieve_limit": N_SIEVE,
  "primes": int(NP),
  "pairs_checked": int(NP - 1),
  "counterexamples_found": int(len(violations) + len(bad_exact) + len(bad_tight)),
  "max_rho": float(rho[i_rho]), "max_rho_at_p": int(P[i_rho]), "max_rho_gap": int(gaps[i_rho]),
  "max_csg": float(csg[i_csg]), "max_csg_at_p": int(P[i_csg]),
  "min_relative_margin_dps50": float(min_margin),
  "exact_integer_checked_upto_n": EXACT_UPTO,
  "record_gaps_in_range": len(rec_idx),
  "T_descent_fraction": desc / tot,
  "certified_range_C1": None if crossover(mpf(1)) is None else float(crossover(mpf(1))),
  "C_required_for_1e7":  float(required_C(mpf('1e7'))),
  "C_required_for_4e18": float(required_C(mpf('4e18'))),
  "B1_over_T_at_1e7": float(math.sqrt(P[-2])*math.log(P[-2]) / T[-2]),
}
print(json.dumps(summary, indent=2))
with open("summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nwrote summary.json")

{
  "target": "RH-conditional-bound",
  "sieve_limit": 10000000,
  "primes": 664579,
  "pairs_checked": 664578,
  "counterexamples_found": 0,
  "max_rho": 0.7604708659164422,
  "max_rho_at_p": 1327,
  "max_rho_gap": 34,
  "max_csg": 0.7025656074174531,
  "max_csg_at_p": 2010733,
  "min_relative_margin_dps50": 6.342262321149653e-07,
  "exact_integer_checked_upto_n": 10000,
  "record_gaps_in_range": 22,
  "T_descent_fraction": 0.5635005544947176,
  "certified_range_C1": null,
  "C_required_for_1e7": 0.004761142189302036,
  "C_required_for_4e18": 2.09047397275023e-08,
  "B1_over_T_at_1e7": 210.15575151043248
}

wrote summary.json


### Verdict `[self-contained, modulo the tags]`

1. **F is corroborated, not proven, below `10^7`.** No counterexample; three independent
   arithmetics agree; the tightest approach reaches `ρ = 0.7605`. A Π₁ statement takes no finite
   certificate — this range result is a *lower bound on where a counterexample can live*, nothing more.

2. **The target claim survives the stress test, and in a stronger form than stated.** `decompose`
   §2.1/§3.2 asserts RH does not reach Firoozbakht. The computation here shows the failure is
   **uniform in the constant**: a bound of shape `C√p·log p` has a *finite* certified range `p*(C)`
   for every `C > 0`, and `required_C(P) ~ log P/√P → 0`. One would need `C ≈ 2·10⁻⁸` merely to match
   an already-computed range, and no positive `C` at all suffices for every `n`. **This conclusion is
   independent of every `[L3-recall]` constant in the notebook** — which matters, because card L11
   records that the RH-conditional gap bound has no ledger row in this run.

3. **A distinction the prose should keep.** RH's `π(x)` error control is *more* than precise enough
   for this problem (§4): it perturbs the bar by `~log²p/(8π√p)`, orders of magnitude below the
   margin in dispute. The obstruction is entirely on the gap side, and is a mismatch of *scale*
   (`√p` vs `log²p`), not of *precision*.

4. **What computation refuted here:** nothing about F. What it refuted is a *hope* — that a smaller
   constant, or a sharper RH input, could bring the conditional route within reach. §3.2 kills that
   quantitatively.

5. **What remains undischarged and was not touched:** P6′ (the maximal-gap reduction) — §2 confirms
   `T` is non-monotone at 56.4% of steps, so the obstruction is real; this notebook does not repair
   it. The `[L3-recall]` constants still need the citation gate.